In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

file_path = "DistilRoBERTa FinBERT LLM emotion .xlsx"
df = pd.read_excel(file_path)

def compute_vif_table(df, x_vars):
    X = df[x_vars].dropna().copy()
    X = sm.add_constant(X)

    vif_table = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })

    return vif_table



In [4]:
finbert_vars = ["FinBERT"]

llm_vars = [
    "LLM_fear",
    "LLM_anger",
    "LLM_disgust",
    "LLM_joy",
    "LLM_sadness",
    "LLM_surprise"
]

distil_vars = [
    "DistilRoBERTa_fear",
    "DistilRoBERTa_anger",
    "DistilRoBERTa_disgust",
    "DistilRoBERTa_joy",
    "DistilRoBERTa_sadness",
    "DistilRoBERTa_surprise"
]

In [5]:
asset_lags = {
    "natural_gas": "natural_gas_return_t-1",
    "oil": "oil_return_t-1",
    "clean": "clean_return_t-1"
}

In [6]:
all_vif_results = []

for asset, lag_var in asset_lags.items():
    model_specs = {
        "Model 1": [lag_var] + finbert_vars,
        "Model 2": [lag_var] + llm_vars,
        "Model 3": [lag_var] + distil_vars,
        "Model 4": [lag_var] + finbert_vars + llm_vars,
        "Model 5": [lag_var] + llm_vars + distil_vars,
        "Model 6": [lag_var] + finbert_vars + llm_vars + distil_vars
    }

    for model_name, x_vars in model_specs.items():
        vif_table = compute_vif_table(df, x_vars)
        vif_table["asset"] = asset
        vif_table["model"] = model_name
        vif_table = vif_table[["asset", "model", "variable", "VIF"]]
        all_vif_results.append(vif_table)

In [7]:
vif_results_18 = pd.concat(all_vif_results, ignore_index=True)

vif_results_18["model_num"] = vif_results_18["model"].str.extract(r"(\d+)").astype(int)
vif_results_18 = vif_results_18.sort_values(
    by=["asset", "model_num", "VIF"],
    ascending=[True, True, False]
).drop(columns="model_num")

print(vif_results_18)

     asset    model               variable         VIF
114  clean  Model 1                  const    2.959342
115  clean  Model 1       clean_return_t-1    1.027774
116  clean  Model 1                FinBERT    1.027774
117  clean  Model 2                  const  126.090475
120  clean  Model 2              LLM_anger    3.142477
..     ...      ...                    ...         ...
112    oil  Model 6  DistilRoBERTa_sadness    1.374965
107    oil  Model 6           LLM_surprise    1.358113
109    oil  Model 6    DistilRoBERTa_anger    1.349926
108    oil  Model 6     DistilRoBERTa_fear    1.340141
100    oil  Model 6         oil_return_t-1    1.136013

[171 rows x 4 columns]
